# Local Transfer Workflow Notebook

This notebook automates the end-to-end local transfer workflow:
1. Set parameters and paths
2. Check/install dependencies and environment
3. Define reusable utility functions
4. Load inputs in batch
5. Validate and preprocess inputs
6. Run baseline and pseudo-label pipeline
7. Add logging, retries, and failure capture
8. Save outputs and build summary reports
9. Run everything from one main cell

In [1]:
# 1) Parameters and paths
from pathlib import Path
import json
import subprocess
import sys
import time
from datetime import datetime

import pandas as pd

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / 'evaluate_local_baseline.py').exists():
    raise FileNotFoundError('Open this notebook from the project root folder.')

LOCAL_DIR = PROJECT_DIR / 'processed_data' / 'local'
PRED_LOCAL = PROJECT_DIR / 'predictions' / 'local'
PRED_LOCAL.mkdir(parents=True, exist_ok=True)

# Core run parameters
BASELINE_SPLIT = 'dev'  # keep test split untouched for final reporting
TEST_SIZE = 0.2
SEED = 42

CREATE_FINAL_HOLDOUT = True
FINAL_HOLDOUT_DIR = LOCAL_DIR / 'final_holdout'

RUN_PSEUDO_ROUND_1 = True
RUN_PSEUDO_ROUND_2 = True
RUN_PSEUDO_ROUND_3 = True
RUN_PSEUDO_ROUND_4 = True

# Round 1 (strictest among the 4 configured passes)
R1 = {
    'run_id': 'P1',
    'round': 1,
    'fraud_min': 0.58,
    'legit_max': 0.24,
    'agreement': 2,
    'max_legit_ratio': 3.0,
}

# Round 2 (relaxed)
R2 = {
    'run_id': 'P2',
    'round': 2,
    'fraud_min': 0.54,
    'legit_max': 0.26,
    'agreement': 2,
    'max_legit_ratio': 3.0,
}

# Round 3 (wider selection, keep class cap looser)
R3 = {
    'run_id': 'P3',
    'round': 3,
    'fraud_min': 0.50,
    'legit_max': 0.30,
    'agreement': 2,
    'max_legit_ratio': 5.0,
}

# Round 4 (widest pass with agreement still fixed at 2)
R4 = {
    'run_id': 'P4',
    'round': 4,
    'fraud_min': 0.48,
    'legit_max': 0.34,
    'agreement': 2,
    'max_legit_ratio': 5.0,
}

print('Project:', PROJECT_DIR)
print('Local data dir:', LOCAL_DIR)
print('Predictions dir:', PRED_LOCAL)
print('Holdout export dir:', FINAL_HOLDOUT_DIR)

Project: c:\Users\micof\Documents\GitHub\e-commerce-fraud-detection
Local data dir: c:\Users\micof\Documents\GitHub\e-commerce-fraud-detection\processed_data\local
Predictions dir: c:\Users\micof\Documents\GitHub\e-commerce-fraud-detection\predictions\local
Holdout export dir: c:\Users\micof\Documents\GitHub\e-commerce-fraud-detection\processed_data\local\final_holdout


In [2]:
# 2) Dependency import/install check
import importlib

# Keep this aligned with imports used by predict.py and baseline/pseudo scripts.
required_modules = [
    'pandas', 'numpy', 'sklearn',
    'torch', 'torchvision', 'transformers',
    'xgboost', 'requests', 'PIL', 'joblib', 'nltk',
]
missing = [m for m in required_modules if importlib.util.find_spec(m) is None]
if missing:
    print('Missing modules:', missing)
    print('Installing missing modules...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', *missing], check=True)
else:
    print('All required modules are available.')

All required modules are available.


In [3]:
# Environment readiness check
print('Python:', sys.version)
print('Python executable:', sys.executable)
print('Timestamp:', datetime.now().isoformat(timespec='seconds'))

required_files = [
    PROJECT_DIR / 'evaluate_local_baseline.py',
    PROJECT_DIR / 'generate_local_pseudolabels.py',
    PROJECT_DIR / 'create_local_final_holdout.py',
    LOCAL_DIR / 'local_labeled_text_dataset.csv',
    LOCAL_DIR / 'local_labeled_image_dataset.csv',
    LOCAL_DIR / 'local_labeled_metadata_dataset.csv',
    LOCAL_DIR / 'local_unlabeled_raw.csv',
    LOCAL_DIR / 'local_unlabeled_text_dataset.csv',
    LOCAL_DIR / 'local_unlabeled_image_dataset.csv',
    LOCAL_DIR / 'local_unlabeled_metadata_dataset.csv',
]
missing_files = [str(p) for p in required_files if not p.exists()]
if missing_files:
    raise FileNotFoundError('Missing required files:\n' + '\n'.join(missing_files))
print('Environment ready.')

Python: 3.13.5 | packaged by Anaconda, Inc. | (main, Jun 12 2025, 16:37:03) [MSC v.1929 64 bit (AMD64)]
Python executable: c:\Users\micof\anaconda3\python.exe
Timestamp: 2026-03-16T17:00:11
Environment ready.


## 3) Reusable Utility Functions

In [4]:
def run_cmd(args, retries=0, retry_wait=2.0):
    """Run command with optional retry, capture stdout/stderr, and return metadata."""
    last_err = None
    for attempt in range(retries + 1):
        t0 = time.time()
        completed = subprocess.run(args, capture_output=True, text=True)
        dt = time.time() - t0
        if completed.returncode == 0:
            return {
                'ok': True,
                'args': args,
                'attempt': attempt + 1,
                'runtime_sec': dt,
                'stdout': completed.stdout,
                'stderr': completed.stderr,
            }
        last_err = {
            'ok': False,
            'args': args,
            'attempt': attempt + 1,
            'runtime_sec': dt,
            'stdout': completed.stdout,
            'stderr': completed.stderr,
            'returncode': completed.returncode,
        }
        if attempt < retries:
            time.sleep(retry_wait)
    return last_err


def read_json(path: Path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

In [5]:
def load_inputs_batch():
    """4) Load key input files into memory as DataFrames."""
    inputs = {
        'labeled_text': LOCAL_DIR / 'local_labeled_text_dataset.csv',
        'labeled_image': LOCAL_DIR / 'local_labeled_image_dataset.csv',
        'labeled_metadata': LOCAL_DIR / 'local_labeled_metadata_dataset.csv',
        'unlabeled_raw': LOCAL_DIR / 'local_unlabeled_raw.csv',
        'unlabeled_text': LOCAL_DIR / 'local_unlabeled_text_dataset.csv',
        'unlabeled_image': LOCAL_DIR / 'local_unlabeled_image_dataset.csv',
        'unlabeled_metadata': LOCAL_DIR / 'local_unlabeled_metadata_dataset.csv',
    }
    data = {k: pd.read_csv(v) for k, v in inputs.items()}
    return data, inputs


def validate_preprocess_inputs(data_dict):
    """5) Basic schema/type/null checks and row-count audit."""
    report_rows = []
    for name, df in data_dict.items():
        cols = set(df.columns)
        has_pid = 'product_id' in cols
        has_label = 'fraud_label' in cols
        null_cells = int(df.isna().sum().sum())
        report_rows.append({
            'dataset': name,
            'rows': int(len(df)),
            'cols': int(df.shape[1]),
            'has_product_id': has_pid,
            'has_fraud_label': has_label,
            'null_cells': null_cells,
        })
    report_df = pd.DataFrame(report_rows)
    return report_df

## 4) Load Inputs in Batch and 5) Validate/Preprocess Inputs

In [6]:
data_dict, input_paths = load_inputs_batch()
validation_report = validate_preprocess_inputs(data_dict)

print('Input files loaded:')
for k, p in input_paths.items():
    print(f'  {k:18s} -> {p}')

print('\nValidation summary:')
validation_report

Input files loaded:
  labeled_text       -> c:\Users\micof\Documents\GitHub\e-commerce-fraud-detection\processed_data\local\local_labeled_text_dataset.csv
  labeled_image      -> c:\Users\micof\Documents\GitHub\e-commerce-fraud-detection\processed_data\local\local_labeled_image_dataset.csv
  labeled_metadata   -> c:\Users\micof\Documents\GitHub\e-commerce-fraud-detection\processed_data\local\local_labeled_metadata_dataset.csv
  unlabeled_raw      -> c:\Users\micof\Documents\GitHub\e-commerce-fraud-detection\processed_data\local\local_unlabeled_raw.csv
  unlabeled_text     -> c:\Users\micof\Documents\GitHub\e-commerce-fraud-detection\processed_data\local\local_unlabeled_text_dataset.csv
  unlabeled_image    -> c:\Users\micof\Documents\GitHub\e-commerce-fraud-detection\processed_data\local\local_unlabeled_image_dataset.csv
  unlabeled_metadata -> c:\Users\micof\Documents\GitHub\e-commerce-fraud-detection\processed_data\local\local_unlabeled_metadata_dataset.csv

Validation summary:


,dataset,rows,cols,has_product_id,has_fraud_label,null_cells
0,labeled_text,349,6,True,True,44
1,labeled_image,349,3,True,True,0
2,labeled_metadata,349,27,True,True,0
3,unlabeled_raw,257,17,True,True,728
4,unlabeled_text,257,6,True,True,472
5,unlabeled_image,257,3,True,True,257
6,unlabeled_metadata,257,27,True,True,257


 # 6) Core automation pipeline (baseline + pseudo rounds)

In [7]:
def run_pipeline():
    run_artifacts = {
        'timestamp': datetime.now().isoformat(timespec='seconds'),
        'steps': [],
        'failures': [],
    }

    # Create/export untouched final-test holdout files once per run.
    if CREATE_FINAL_HOLDOUT:
        holdout_cmd = [
            sys.executable, 'create_local_final_holdout.py',
            '--test-size', str(TEST_SIZE),
            '--seed', str(SEED),
            '--output-dir', str(FINAL_HOLDOUT_DIR),
        ]
        res = run_cmd(holdout_cmd, retries=0)
        run_artifacts['steps'].append({'name': 'prepare_final_holdout', **res})
        if not res['ok']:
            run_artifacts['failures'].append('prepare_final_holdout')
            return run_artifacts

    # Baseline
    baseline_cmd = [
        sys.executable, 'evaluate_local_baseline.py',
        '--split', BASELINE_SPLIT,
        '--test-size', str(TEST_SIZE),
        '--seed', str(SEED),
    ]
    res = run_cmd(baseline_cmd, retries=0)
    run_artifacts['steps'].append({'name': 'baseline', **res})
    if not res['ok']:
        run_artifacts['failures'].append('baseline')
        return run_artifacts

    # Pseudo rounds
    pseudo_rounds = [
        ('pseudo_round_1', RUN_PSEUDO_ROUND_1, R1),
        ('pseudo_round_2', RUN_PSEUDO_ROUND_2, R2),
        ('pseudo_round_3', RUN_PSEUDO_ROUND_3, R3),
        ('pseudo_round_4', RUN_PSEUDO_ROUND_4, R4),
    ]
    for step_name, enabled, cfg in pseudo_rounds:
        if not enabled:
            continue
        pseudo_cmd = [
            sys.executable, 'generate_local_pseudolabels.py',
            '--run-id', cfg['run_id'],
            '--round', str(cfg['round']),
            '--fraud-min', str(cfg['fraud_min']),
            '--legit-max', str(cfg['legit_max']),
            '--agreement', str(cfg['agreement']),
            '--max-legit-ratio', str(cfg['max_legit_ratio']),
        ]
        res = run_cmd(pseudo_cmd, retries=1, retry_wait=3.0)
        run_artifacts['steps'].append({'name': step_name, **res})
        if not res['ok']:
            run_artifacts['failures'].append(step_name)

    return run_artifacts

## 7) Logging, Error Handling, Retries and 8) Output/Report Generation

In [8]:
run_report = run_pipeline()

# Save run report JSON
run_report_path = PRED_LOCAL / f"run_report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
with open(run_report_path, 'w', encoding='utf-8') as f:
    json.dump(run_report, f, indent=2)

print('Run report saved:', run_report_path)
print('Failures:', run_report['failures'])

# Load output summaries when available
baseline_metrics_path = PRED_LOCAL / 'local_baseline_metrics.json'
pseudo_stats_path = PRED_LOCAL / 'local_unlabeled_pseudo_stats.json'
log_path = PRED_LOCAL / 'experiment_log.csv'

if baseline_metrics_path.exists():
    baseline = read_json(baseline_metrics_path)
    print('\nBaseline summary:')
    for k in ['text', 'image', 'metadata', 'ensemble']:
        m = baseline.get('metrics', {}).get(k, {})
        if 'f1' in m:
            print(f"  {k:8s} F1={m['f1']:.4f} AUC={m['roc_auc']:.4f} n={m['samples']}")

if pseudo_stats_path.exists():
    pseudo = read_json(pseudo_stats_path)
    c = pseudo.get('counts', {})
    print('\nPseudo-label summary:')
    print('  unlabeled_total:', c.get('unlabeled_total'))
    print('  filtered_total :', c.get('filtered_total'))
    print('  filtered_fraud :', c.get('filtered_fraud'))
    print('  filtered_legit :', c.get('filtered_legit'))

if log_path.exists():
    exp_log = pd.read_csv(log_path)
    print('\nExperiment log tail:')
    display(exp_log.tail(10))

Run report saved: c:\Users\micof\Documents\GitHub\e-commerce-fraud-detection\predictions\local\run_report_20260316_171837.json
Failures: []

Baseline summary:
  text     F1=0.0000 AUC=0.5048 n=279
  image    F1=0.1220 AUC=0.4359 n=279
  metadata F1=0.3607 AUC=0.8656 n=279
  ensemble F1=0.1818 AUC=0.7095 n=279

Pseudo-label summary:
  unlabeled_total: 257
  filtered_total : 96
  filtered_fraud : 16
  filtered_legit : 80

Experiment log tail:


,run_timestamp,experiment,split,test_size,seed,rows_scored,fraud_count,text_f1,text_auc,image_f1,...,ensemble_auc,ensemble_threshold,run_id,pseudo_fraud_min,pseudo_legit_max,pseudo_agreement,pseudo_max_legit_ratio,pseudo_kept_total,pseudo_kept_fraud,pseudo_kept_legit
6,2026-03-16T15:40:10,pseudo_label_round_99,unlabeled,NaN,NaN,257,NaN,NaN,NaN,NaN,...,NaN,NaN,TEST_ARCHIVE,0.58,0.24,2.0,3.0,27.0,10.0,17.0
7,2026-03-16T16:30:15,local_baseline_pre_transfer_dev,dev,0.2,42.0,279,50.0,0.0,0.504803,0.121951,...,0.70952,0.515,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,2026-03-16T16:32:23,pseudo_label_round_1,unlabeled,NaN,NaN,257,NaN,NaN,NaN,NaN,...,NaN,NaN,P1,0.58,0.24,2.0,3.0,27.0,10.0,17.0
9,2026-03-16T16:34:05,pseudo_label_round_2,unlabeled,NaN,NaN,257,NaN,NaN,NaN,NaN,...,NaN,NaN,P2,0.54,0.26,2.0,3.0,50.0,13.0,37.0
10,2026-03-16T16:35:54,pseudo_label_round_3,unlabeled,NaN,NaN,257,NaN,NaN,NaN,NaN,...,NaN,NaN,P3,0.50,0.30,2.0,3.0,64.0,16.0,48.0
11,2026-03-16T17:02:14,local_baseline_pre_transfer_dev,dev,0.2,42.0,279,50.0,0.0,0.504803,0.121951,...,0.70952,0.515,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12,2026-03-16T17:04:51,pseudo_label_round_1,unlabeled,NaN,NaN,257,NaN,NaN,NaN,NaN,...,NaN,NaN,P1,0.58,0.24,2.0,3.0,27.0,10.0,17.0
13,2026-03-16T17:09:42,pseudo_label_round_2,unlabeled,NaN,NaN,257,NaN,NaN,NaN,NaN,...,NaN,NaN,P2,0.54,0.26,2.0,3.0,50.0,13.0,37.0
14,2026-03-16T17:14:17,pseudo_label_round_3,unlabeled,NaN,NaN,257,NaN,NaN,NaN,NaN,...,NaN,NaN,P3,0.50,0.30,2.0,5.0,96.0,16.0,80.0
15,2026-03-16T17:18:35,pseudo_label_round_4,unlabeled,NaN,NaN,257,NaN,NaN,NaN,NaN,...,NaN,NaN,P4,0.48,0.34,2.0,5.0,96.0,16.0,80.0


## 9) Single-Click Re-Run

To re-run the whole workflow:
1. Update parameters in Cell 2
2. Run all cells from top to bottom

This keeps split/test settings, pseudo-label thresholds, and run reports consistent across experiments.